In [0]:
%run ../log/00_audit_logging

In [0]:
# Databricks notebook source
# MAGIC %run ../log/audit_logging

# COMMAND ----------

# BRONZE -> SILVER
# Parse raw JSON, cast types, validate, dedupe, quarantine bad rows, MERGE into Silver.

from datetime import datetime
from functools import reduce
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType

BRONZE = "workspace.tmdb.bronze_movies"
SILVER = "workspace.tmdb.silver_movies"
QUARANTINE = "workspace.tmdb.quarantine_movies"

# COMMAND ----------

# Parameters
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"], "Load type")
dbutils.widgets.text("batch_ids", "", "Batch IDs, comma separated (blank = latest batch)")
dbutils.widgets.text("pulled_from", "", "OR pulled_at from (YYYY-MM-DD)")
dbutils.widgets.text("pulled_to", "", "OR pulled_at to (YYYY-MM-DD)")

load_type   = dbutils.widgets.get("load_type")
batch_ids   = [b.strip() for b in dbutils.widgets.get("batch_ids").split(",") if b.strip()]
pulled_from = dbutils.widgets.get("pulled_from").strip()
pulled_to   = dbutils.widgets.get("pulled_to").strip()

# COMMAND ----------

# Create Silver table with explicit types. Primary key = movie_id (enforced by MERGE).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER} (
    movie_id          INT,
    title             STRING,
    original_title    STRING,
    overview          STRING,
    release_date      DATE,
    popularity        DOUBLE,
    vote_average      DOUBLE,
    vote_count        LONG,
    budget            LONG,
    revenue           LONG,
    runtime           INT,
    status            STRING,
    tagline           STRING,
    source_collection STRING,
    source_file       STRING,
    pulled_at         DATE,
    batch_id          STRING,
    load_timestamp    TIMESTAMP
) USING DELTA
""")

# COMMAND ----------

# Explicit schema for the movie JSON. Every field is read as STRING first, then cast safely.
# That way a type change in the source (e.g. id becomes "abc") is detected, not silently lost.
movie_fields = ["id", "title", "original_title", "overview", "release_date", "popularity",
                "vote_average", "vote_count", "budget", "revenue", "runtime", "status", "tagline"]
raw_movie_schema = StructType([StructField(f, StringType(), True) for f in movie_fields])

# COMMAND ----------

start_time = datetime.now()
batch_label = "n/a"
param_text = "n/a"

try:
    # ---- 1. Pick which Bronze rows to process --------------------------------
    bronze = spark.table(BRONZE)
    if load_type == "full":
        bronze = bronze.filter("source_collection = 'full_load'")
    else:
        bronze = bronze.filter("source_collection IN ('trending', 'changed')")

    if batch_ids:
        bronze = bronze.filter(F.col("batch_id").isin(batch_ids))
    elif pulled_from and pulled_to:
        bronze = bronze.filter(F.col("pulled_at").between(pulled_from, pulled_to))
    else:
        latest = bronze.orderBy(F.col("load_timestamp").desc()).select("batch_id").first()
        if latest is not None:
            bronze = bronze.filter(F.col("batch_id") == latest[0])

    batches = [r[0] for r in bronze.select("batch_id").distinct().collect()]
    batch_label = ",".join(batches) if batches else "none"
    param_text = f"batch_ids={batch_label} pulled_from={pulled_from} pulled_to={pulled_to}"
    print("Processing batches:", batch_label)

    # ---- 2. Parse + cast ------------------------------------------------------
    df = bronze.withColumn("p", F.from_json("raw_record", raw_movie_schema))
    for f in movie_fields:   # empty strings -> null
        df = df.withColumn("r_" + f, F.when(F.trim(F.col("p." + f)) != "", F.col("p." + f)))

    df = (df
          .withColumn("movie_id",     F.expr("try_cast(r_id as int)"))
          .withColumn("title",        F.col("r_title"))
          .withColumn("original_title", F.col("r_original_title"))
          .withColumn("overview",     F.col("r_overview"))
          .withColumn("release_date", F.expr("try_to_date(r_release_date, 'yyyy-MM-dd')"))
          .withColumn("popularity",   F.expr("try_cast(r_popularity as double)"))
          .withColumn("vote_average", F.expr("try_cast(r_vote_average as double)"))
          .withColumn("vote_count",   F.expr("try_cast(r_vote_count as bigint)"))
          .withColumn("budget",       F.expr("try_cast(r_budget as bigint)"))
          .withColumn("revenue",      F.expr("try_cast(r_revenue as bigint)"))
          .withColumn("runtime",      F.expr("try_cast(r_runtime as int)"))
          .withColumn("status",       F.col("r_status"))
          .withColumn("tagline",      F.col("r_tagline"))
          .withColumn("pulled_at",    F.expr("try_to_date(pulled_at, 'yyyy-MM-dd')")))

    # ---- 3. Validate ----------------------------------------------------------
    # a value existed in the JSON but could not be cast -> the data type changed
    typed_pairs = [("r_id", "movie_id"), ("r_release_date", "release_date"),
                   ("r_popularity", "popularity"), ("r_vote_average", "vote_average"),
                   ("r_vote_count", "vote_count"), ("r_budget", "budget"),
                   ("r_revenue", "revenue"), ("r_runtime", "runtime")]
    type_changed = reduce(lambda a, b: a | b,
                          [F.col(r).isNotNull() & F.col(t).isNull() for r, t in typed_pairs])

    reject_reason = (
        F.when(F.col("movie_id").isNull(), "movie_id missing or not an integer")
         .when(F.col("title").isNull(), "title missing")
         .when(type_changed, "wrong data type in a field (schema drift)")
         .when((F.col("vote_average") < 0) | (F.col("vote_average") > 10), "vote_average outside 0-10")
         .when((F.col("runtime") < 0) | (F.col("revenue") < 0) | (F.col("budget") < 0),
               "negative runtime, revenue or budget")
    )
    df = df.withColumn("reject_reason", reject_reason)

    bad_df = df.filter("reject_reason IS NOT NULL")
    good_df = df.filter("reject_reason IS NULL")

    # ---- 4. Dedupe: keep one row per movie_id (latest pull, prefer rows with details)
    w = Window.partitionBy("movie_id").orderBy(
        F.col("pulled_at").desc(), F.col("runtime").isNotNull().desc())
    silver_df = (good_df
                 .withColumn("rn", F.row_number().over(w))
                 .filter("rn = 1")
                 .select("movie_id", "title", "original_title", "overview", "release_date",
                         "popularity", "vote_average", "vote_count", "budget", "revenue",
                         "runtime", "status", "tagline", "source_collection", "source_file",
                         "pulled_at", "batch_id", F.current_timestamp().alias("load_timestamp")))

    # ---- 5. Quarantine bad rows (re-run safe) ---------------------------------
    if batches:
        id_list = ",".join(f"'{b}'" for b in batches)
        spark.sql(f"DELETE FROM {QUARANTINE} WHERE layer = 'Bronze-to-Silver' AND batch_id IN ({id_list})")

    (bad_df.select(F.lit("Bronze-to-Silver").alias("layer"), "batch_id", "source_file",
                   "reject_reason", "raw_record", F.current_timestamp().alias("load_timestamp"))
           .write.format("delta").mode("append").saveAsTable(QUARANTINE))
    rows_quarantined = bad_df.count()

    # ---- 6. MERGE INTO Silver (idempotent upsert) ------------------------------
    rows_inserted = rows_updated = 0
    if batches:
        silver_df.createOrReplaceTempView("silver_updates")
        spark.sql(f"""
        MERGE INTO {SILVER} t
        USING silver_updates s
        ON t.movie_id = s.movie_id
        WHEN MATCHED AND s.pulled_at >= t.pulled_at THEN UPDATE SET
            t.title             = coalesce(s.title, t.title),
            t.original_title    = coalesce(s.original_title, t.original_title),
            t.overview          = coalesce(s.overview, t.overview),
            t.release_date      = coalesce(s.release_date, t.release_date),
            t.popularity        = coalesce(s.popularity, t.popularity),
            t.vote_average      = coalesce(s.vote_average, t.vote_average),
            t.vote_count        = coalesce(s.vote_count, t.vote_count),
            t.budget            = coalesce(s.budget, t.budget),
            t.revenue           = coalesce(s.revenue, t.revenue),
            t.runtime           = coalesce(s.runtime, t.runtime),
            t.status            = coalesce(s.status, t.status),
            t.tagline           = coalesce(s.tagline, t.tagline),
            t.source_collection = s.source_collection,
            t.source_file       = s.source_file,
            t.pulled_at         = s.pulled_at,
            t.batch_id          = s.batch_id,
            t.load_timestamp    = s.load_timestamp
        WHEN NOT MATCHED THEN INSERT *
        """)

        # inserted / updated counts come from the Delta history of the MERGE
        metrics = spark.sql(f"DESCRIBE HISTORY {SILVER} LIMIT 1").collect()[0]["operationMetrics"]
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

    log_pipeline_execution("Bronze-to-Silver", load_type, batch_label, param_text,
                           start_time, datetime.now(), "SUCCESS",
                           rows_inserted=rows_inserted, rows_updated=rows_updated,
                           rows_quarantined=rows_quarantined)

except Exception as e:
    log_pipeline_execution("Bronze-to-Silver", load_type, batch_label, param_text,
                           start_time, datetime.now(), "FAILURE", error_msg=e)
    raise

In [0]:
%sql
SELECT movie_id, COUNT(*) 
FROM (SELECT get_json_object(raw_record,'$.id') AS movie_id FROM workspace.tmdb.bronze_movies)
GROUP BY movie_id HAVING COUNT(*) > 1
--checking dups

In [0]:
%sql
SELECT batch_id, COUNT(*) AS rows_in_batch
FROM workspace.tmdb.bronze_movies
GROUP BY batch_id

In [0]:
%sql
DELETE FROM workspace.tmdb.bronze_movies
WHERE batch_id = '20261009_153201'

In [0]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT movie_id) AS distinct_ids
FROM workspace.tmdb.silver_movies
--total rows in silver